# RetailMind AI — Day 15: diffusion marketing imagery

Day 14 gave the catalog a voice. Day 15 gives it a camera. We turn the same product metadata into **on-brand studio, lifestyle and social images** with Stable Diffusion.

This notebook reads the saved prompt library, shows the positive/negative prompts and guidance scale, and runs the generator with a fake pipeline so it works without a GPU or a 4 GB model download. Real generation is one CLI command away on a GPU machine.

In [1]:
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(ROOT)
sys.path.insert(0, ROOT)
print("project folder:", os.path.basename(ROOT))

project folder: retailmind-ai


## 1. The prompt library

`data/prompt_library.json` holds nine briefs: three products times three styles. It is built from `product_catalog.json`, so there is no separate dataset to maintain.

In [2]:
import json

library = json.load(open("data/prompt_library.json", encoding="utf-8"))
specs = library["prompts"]
print(library["source"], "|", library["model"], "|", len(specs), "prompts")
for spec in specs[:3]:
    print(spec["product_id"], spec["style"], spec["seed"], spec["setting"])

product_catalog.json | runwayml/stable-diffusion-v1-5 | 9 prompts
RM-AUDIO-01 catalog 101 a tidy desk with a laptop and notebook
RM-AUDIO-01 lifestyle 112 a tidy desk with a laptop and notebook
RM-AUDIO-01 social 124 a tidy desk with a laptop and notebook


## 2. One complete brief

The positive prompt carries the product name, its features, the setting and the brand voice. The negative prompt lists what must not appear. Guidance scale `7.5` says how closely the model should follow the brief.

In [3]:
first = specs[0]
print(first["prompt"])
print("negative:", first["negative_prompt"])
print(f"guidance {first['guidance_scale']} | steps {first['steps']} | {first['width']}x{first['height']}")

Professional product photograph of RetailMind Studio Wireless Headphones, an audio product. Over-ear headphones tuned for balanced sound with soft memory-foam ear cushions. They weigh 250 g, fold flat for travel and include a USB-C charging cable. Key features: 40-hour battery life with quick charge; Active noise cancellation; Bluetooth 5.3 multipoint pairing; Replaceable ear cushions and folding case. clean studio product photography, soft neutral background, diffused light, sharp focus, centered product. Setting: a tidy desk with a laptop and notebook. Brand tone: Warm, concise and practical. Speak as a helpful RetailMind product specialist, never as a robot. Use short sentences and plain language. Premium retail campaign, realistic materials, sharp focus, no text, no logo.
negative: blurry, low quality, low resolution, distorted, deformed, extra limbs, watermark, text, logo, brand name, busy background, clutter, oversaturated, harsh shadows, wrong product, mangled details
guidance 7

## 3. Three styles, one brand

Only the style phrase changes between channels. The product facts and brand voice stay fixed, which is what keeps the output recognisably RetailMind.

In [4]:
from src.creative.prompts import STYLE_PROMPTS

for style, text in STYLE_PROMPTS.items():
    print(style, text)

catalog clean studio product photography, soft neutral background, diffused light, sharp focus, centered product
lifestyle warm authentic lifestyle photography, natural daylight, real home setting, relaxed and practical mood
social bright modern editorial flat lay, clean negative space, cheerful palette, social-media ready composition


## 4. Seeds make approved assets reproducible

The seed fixes the starting noise. Reusing a prompt and its seed should give the same picture, so an approval is not lost. The nine seeds below are stable because they come from a product base plus a small style offset.

In [5]:
from src.creative.prompts import resolve_seed

print([spec["seed"] for spec in specs])
print(resolve_seed("RM-AUDIO-01", "catalog") == resolve_seed("RM-AUDIO-01", "catalog"))

[101, 112, 124, 202, 213, 225, 303, 314, 326]
True


## 5. The generator call, without a GPU

`MarketingImageGenerator` lazily loads the diffusers pipeline only when an image is requested. Here we inject a fake pipeline and a fake seed factory, so the cell shows the exact arguments and writes a tiny placeholder PNG in a temporary folder.

In [6]:
import tempfile
from pathlib import Path
from types import SimpleNamespace

from src.creative import MarketingImageGenerator

class FakeImage:
    def save(self, path):
        Path(path).write_bytes(b"fake png")

class FakePipeline:
    def __init__(self):
        self.calls = []

    def __call__(self, **kwargs):
        self.calls.append(kwargs)
        return SimpleNamespace(images=[FakeImage()])

pipeline = FakePipeline()
with tempfile.TemporaryDirectory() as tmp:
    creative = MarketingImageGenerator(pipeline=pipeline, output_folder=tmp, device="cpu")
    record = creative.generate(first, generator_factory=lambda seed, device: object())
    print("saved PNG exists:", Path(record["output"]).exists(), "| device:", record["device"])
print("record keys:", sorted(record.keys()))

saved PNG exists: True | device: cpu
record keys: ['device', 'guidance_scale', 'height', 'model', 'negative_prompt', 'output', 'product_id', 'product_name', 'prompt', 'seed', 'setting', 'steps', 'style', 'width']


In [7]:
call = pipeline.calls[0]
print("pipeline args:", sorted(call.keys()))
print("negative prompt used:", call["negative_prompt"] == first["negative_prompt"])
print(f"guidance: {call['guidance_scale']} | steps: {call['num_inference_steps']} | size: {call['width']} {call['height']}")

pipeline args: ['generator', 'guidance_scale', 'height', 'negative_prompt', 'num_inference_steps', 'prompt', 'width']
negative prompt used: True
guidance: 7.5 | steps: 30 | size: 512 512


## Try it yourself

From the project root:

```bash
python -m src.creative prompts --style social
python -m src.creative generate --dry-run --product RM-AUDIO-01 --style catalog
python -m src.creative generate --product RM-AUDIO-01 --style catalog
python -m src.creative generate --all-styles
python -m unittest discover -s tests
```

The first two commands and the tests work on any laptop. The real `generate` command needs a GPU runtime and downloads about 4 GB on first use. It writes PNGs to `reports/marketing/` and a prompt/seed manifest to `reports/marketing_assets.json`; both are gitignored.

**Be honest about the limits:** 30 steps at 512×512 is slow on CPU, diffusion can still distort small product details, and a human should approve assets before publication. Next iteration adds img2img variations and a short promo video clip.